In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
v1

array([ 2.13903859e-02, -7.39799738e-02,  1.42069475e-03,  2.13816315e-02,
        2.45113391e-02,  3.15582603e-02, -1.10839710e-01, -1.05017453e-01,
       -6.18258938e-02, -6.42312970e-03,  3.72394198e-03,  9.06393528e-02,
       -9.49937198e-03,  6.53976947e-02,  1.10946735e-02, -2.10097395e-02,
       -3.35125700e-02, -4.31677736e-02,  9.96346213e-03,  1.41969677e-02,
       -6.40415177e-02, -7.04179332e-03, -7.91188180e-02,  5.80030978e-02,
        1.30213588e-03,  4.19733720e-03,  5.70979156e-02,  6.39447868e-02,
        2.49903090e-02, -3.95876467e-02, -3.79506387e-02,  2.70394497e-02,
        1.79423485e-02,  1.72272474e-02,  3.43311429e-02,  9.29058157e-03,
        5.86054958e-02, -4.97789867e-02, -5.05369157e-03,  4.34328243e-02,
       -1.56622920e-02, -2.97534615e-02, -5.13324142e-03,  5.13414778e-02,
        6.16064621e-03,  6.86980411e-02, -1.29505135e-02, -5.61938696e-02,
       -1.08265216e-02,  5.96683659e-02,  5.29939905e-02, -3.42754982e-02,
       -4.15274203e-02, -

In [5]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [6]:
# compare with dot product
v1.dot(dv)
# close from document

np.float32(0.32332397)

In [7]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [8]:
v2.dot(dv)
# far away from document

np.float32(0.019730574)

## Load and embedded FAQ dataset

In [9]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for directory in (current, *current.parents):
    if (directory / "src" / "ingest.py").is_file():
        sys.path.insert(0, str(directory))
        break
else:
    raise FileNotFoundError("Cannot find src/ingest.py")

from src.ingest import load_faq_data

documents = load_faq_data()

In [10]:
documents[0]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 'doc_id': '9e508f2212'}

In [11]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [12]:
texts[0]

"Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."

1200 texts can take very long => batches

In [13]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [14]:
import numpy as np
X = np.array(vectors)

- rows == document (vectors)
- columns == vector dim 

In [15]:
m = X.shape

In [16]:
print(f"Total documents: {m[0]} | Vector dim: {m[1]}")

Total documents: 1350 | Vector dim: 384


## Vector search

In [17]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [18]:
# Compute scalar dot over doc matrix
scores = X.dot(v_query) # similar to this code: scores = [v_query.dot(X[i]) for i in range(len(X))] but not optimized

In [19]:
# we obtain un mark by doc
scores

array([ 0.48740587,  0.2099193 ,  0.762941  , ..., -0.08637964,
        0.03759784, -0.03037035], shape=(1350,), dtype=float32)

In [20]:
# best match
idx = np.argmax(scores)

print(idx, scores[idx])
print(documents[idx])

2 0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}


document 2 with score ~0,76

In [21]:
# we want top-5
top5 = np.argsort(scores)[-5:]
# the first are the smaller score so we invert
top5 = top5[::-1]
print("top5 idx: ", top5)
print("scores: ", scores[top5])

top5 idx:  [  2 625 907 538   7]
scores:  [0.762941   0.7579372  0.7192131  0.6536311  0.56009984]


In [22]:
top5 = np.argsort(-scores)[:5]
top5

array([  2, 625, 907, 538,   7])

In [23]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

### Vector Search with minsearch

#### Create an Index

In [26]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

#### Searching

In [28]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [29]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

#### Filtering

In [30]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [31]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

## RAG with Vector Search